In [0]:
%sql

-- Sales by region and category, pre-sorted by total region revenue (highest to lowest)
WITH region_totals AS (
  SELECT 
    c.`country` AS region,
    SUM(f.`sales_amount`) AS region_total_revenue
  FROM `workspace`.`gold`.`fact_sales` f
  JOIN `workspace`.`gold`.`dim_customers` c ON f.`customer_key` = c.`customer_key`
  WHERE f.`order_date` IS NOT NULL
    AND c.`country` IS NOT NULL
    AND c.`country` != 'n/a'
  GROUP BY c.`country`
),
region_category_sales AS (
  SELECT 
    c.`country` AS region,
    p.`category`,
    SUM(f.`sales_amount`) AS total_revenue,
    rt.region_total_revenue
  FROM `workspace`.`gold`.`fact_sales` f
  JOIN `workspace`.`gold`.`dim_customers` c ON f.`customer_key` = c.`customer_key`
  JOIN `workspace`.`gold`.`dim_products` p ON f.`product_key` = p.`product_key`
  JOIN region_totals rt ON c.`country` = rt.region
  WHERE f.`order_date` IS NOT NULL
    AND c.`country` IS NOT NULL
    AND c.`country` != 'n/a'
    AND p.`category` IS NOT NULL
  GROUP BY c.`country`, p.`category`, rt.region_total_revenue
)
SELECT 
  region,
  category,
  total_revenue
FROM region_category_sales
ORDER BY region_total_revenue DESC, region, category
